# L1.3 · Transaction features

## What are we doing?

We will turn several event rows into one row per item, then check what the new values mean. This makes event data easier to use in a model.

Run one cell at a time with Shift+Enter. Read the explanation before running each cell.

## Setup

Load the tools used by these examples.

In [ ]:
import numpy as np
import pandas as pd

## 1. Load and compare memory

**What and why.** Categories can reduce memory when text values repeat.

**Look for.** The same information can use different amounts of memory.

In [ ]:
events = pd.DataFrame({'id': ['a','a','b','b'], 'type': ['x','x','y','y'], 'amount': [10,20,30,40]})
print(events)
print('Text bytes:', events['type'].memory_usage(deep=True))
print('Category bytes:', events['type'].astype('category').memory_usage(deep=True))

## 2. Aggregate event rows

**What and why.** We calculate count and total for each ID.

**Look for.** A has two events with total 30. B has one event with total 30.

In [ ]:
events = pd.DataFrame({'id': ['a','a','b'], 'amount': [10,20,30]})
print(events.groupby('id')['amount'].agg(['count','sum']))

## 3. Spend mix and trend

**What and why.** We compare each category's share and the totals in two periods.

**Look for.** Shares add to one and period totals show change over time.

In [ ]:
events = pd.DataFrame({'type': ['x','y','x'], 'period': ['early','early','late'], 'amount': [10,30,20]})
print('Shares:', (events.groupby('type')['amount'].sum() / events['amount'].sum()).round(2))
print('Period totals:', events.groupby('period')['amount'].sum())

## 4. Join and handle no activity

**What and why.** An item with no events needs a flag; a missing average is not a measured zero.

**Look for.** C has no activity and its total is filled with zero.

In [ ]:
items = pd.DataFrame({'id': ['a','b','c']})
totals = pd.DataFrame({'id': ['a','b'], 'total': [30,40]})
joined = items.merge(totals, on='id', how='left')
joined['has_activity'] = joined['total'].notna()
joined['total'] = joined['total'].fillna(0)
print(joined)

## 5. Compare a feature with random noise

**What and why.** We compare how much a signal and random values relate to a target.

**Look for.** The signal is perfectly related. A random feature can still correlate by chance.

In [ ]:
rng = np.random.default_rng(2)
target = pd.Series([1,2,3,4,5,6])
signal = pd.Series([1,2,3,4,5,6])
noise = pd.Series(rng.normal(size=6))
print('Signal correlation:', signal.corr(target))
print('Noise correlation:', round(noise.corr(target), 2))

## 6. Turn an image into numbers

**What and why.** An image can be stored as a grid of pixel values.

**Look for.** A 2 by 2 grid becomes four pixel values.

In [ ]:
image = np.array([[0,255],[255,0]])
print('Image:', image)
print('Pixels:', image.ravel())

## 7. Describe a data contract

**What and why.** We record column names and meanings so another step knows what it receives.

**Look for.** Each saved feature has an explanation.

In [ ]:
contract = {'id': 'unique identifier', 'total': 'sum of activity', 'has_activity': 'whether events exist'}
for column, meaning in contract.items():
    print(column, '-', meaning)

## Practise

Change one input value in a cell, run it again, and explain the result. Run earlier cells again if you restarted the kernel.